# Assignment 1: Convolution from Scratch

## Overview

You will implement the 2D convolution operation from first principles using only NumPy. No
`scipy`, no PyTorch convolution functions. This forces you to understand every detail of the
convolution operation: the sliding window, multi-channel computation, stride, padding, and
the backward pass.

### Learning Objectives
- Implement single-channel and multi-channel 2D convolution from scratch
- Implement the backward pass (gradient computation) for convolution
- Benchmark your implementation against PyTorch's optimized `nn.Conv2d`
- Visualize learned filters from a pretrained CNN

### Key Equations

**Output size formula:**

$$H_{out} = \left\lfloor \frac{H + 2 \times \text{padding} - K_h}{\text{stride}} \right\rfloor + 1$$

$$W_{out} = \left\lfloor \frac{W + 2 \times \text{padding} - K_w}{\text{stride}} \right\rfloor + 1$$

**Estimated time:** 8-12 hours

---
## Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import matplotlib.pyplot as plt
import time
import sys
sys.path.insert(0, '../../')

from shared_utils.common import set_seed, get_device

set_seed(42)
device = get_device()
print(f"Using device: {device}")

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

### Helper Functions

In [ ]:
def compute_output_size(input_size: int, kernel_size: int, stride: int = 1, padding: int = 0) -> int:
    """Compute the output dimension for one spatial axis."""
    return (input_size + 2 * padding - kernel_size) // stride + 1


def pad_input(x: np.ndarray, padding: int) -> np.ndarray:
    """Zero-pad a 2D or 4D array along spatial dimensions.
    
    Args:
        x: Array of shape (H, W) or (N, C, H, W)
        padding: Number of zeros to add on each side
        
    Returns:
        Padded array
    """
    if padding == 0:
        return x
    if x.ndim == 2:
        return np.pad(x, ((padding, padding), (padding, padding)), mode='constant')
    elif x.ndim == 4:
        return np.pad(x, ((0, 0), (0, 0), (padding, padding), (padding, padding)), mode='constant')
    else:
        raise ValueError(f"Expected 2D or 4D input, got {x.ndim}D")


# Pre-defined edge detection kernels for testing
KERNELS = {
    'horizontal_edge': np.array([[-1, -1, -1],
                                 [ 0,  0,  0],
                                 [ 1,  1,  1]], dtype=np.float32),
    'vertical_edge':   np.array([[-1, 0, 1],
                                 [-1, 0, 1],
                                 [-1, 0, 1]], dtype=np.float32),
    'sobel_x':         np.array([[-1, 0, 1],
                                 [-2, 0, 2],
                                 [-1, 0, 1]], dtype=np.float32),
    'sobel_y':         np.array([[-1, -2, -1],
                                 [ 0,  0,  0],
                                 [ 1,  2,  1]], dtype=np.float32),
    'sharpen':         np.array([[ 0, -1,  0],
                                 [-1,  5, -1],
                                 [ 0, -1,  0]], dtype=np.float32),
    'blur':            np.ones((3, 3), dtype=np.float32) / 9.0,
}

print("Helper functions loaded.")
print(f"Available kernels: {list(KERNELS.keys())}")

---
## Part 1: Single-Channel Convolution (Forward Pass)

Implement a function `conv2d_forward` that performs 2D convolution (cross-correlation) on a
single-channel input with a single filter.

**Requirements:**
1. Support arbitrary kernel sizes (not just 3x3)
2. Support stride > 1
3. Support zero-padding
4. Compute the output using explicit loops (understand the algorithm first)

**Note:** PyTorch implements cross-correlation (no kernel flip). Your implementation should
also use cross-correlation to match.

In [ ]:
def conv2d_forward(input: np.ndarray, kernel: np.ndarray, stride: int = 1, padding: int = 0) -> np.ndarray:
    """Perform 2D convolution (cross-correlation) on a single-channel input.
    
    Args:
        input: numpy array of shape (H, W)
        kernel: numpy array of shape (K_h, K_w)
        stride: int, stride of the convolution
        padding: int, zero-padding added to both sides
        
    Returns:
        output: numpy array of shape (H_out, W_out)
        where H_out = (H + 2*padding - K_h) // stride + 1
              W_out = (W + 2*padding - K_w) // stride + 1
    """
    # YOUR CODE HERE
    # Steps:
    # 1. Pad the input if padding > 0
    # 2. Compute output dimensions using the formula
    # 3. Slide the kernel across the input, computing element-wise products and summing
    raise NotImplementedError("Implement conv2d_forward")

### Test: Single-Channel Convolution

Verify your implementation against `torch.nn.functional.conv2d` with 5 configurations.

In [ ]:
def test_conv2d_forward():
    """Test single-channel convolution against PyTorch."""
    np.random.seed(42)
    input_np = np.random.randn(8, 8).astype(np.float32)
    
    configs = [
        {"kernel_size": 3, "stride": 1, "padding": 0, "desc": "3x3, stride 1, no padding"},
        {"kernel_size": 3, "stride": 1, "padding": 1, "desc": "3x3, stride 1, same padding"},
        {"kernel_size": 5, "stride": 1, "padding": 2, "desc": "5x5, stride 1, same padding"},
        {"kernel_size": 3, "stride": 2, "padding": 1, "desc": "3x3, stride 2, padding 1"},
        {"kernel_size": 7, "stride": 2, "padding": 3, "desc": "7x7, stride 2, padding 3"},
    ]
    
    for cfg in configs:
        k = cfg["kernel_size"]
        kernel_np = np.random.randn(k, k).astype(np.float32)
        
        # Your implementation
        output_np = conv2d_forward(input_np, kernel_np, stride=cfg["stride"], padding=cfg["padding"])
        
        # PyTorch reference
        input_torch = torch.from_numpy(input_np).unsqueeze(0).unsqueeze(0)
        kernel_torch = torch.from_numpy(kernel_np).unsqueeze(0).unsqueeze(0)
        output_torch = F.conv2d(input_torch, kernel_torch, stride=cfg["stride"], padding=cfg["padding"]).squeeze().numpy()
        
        max_diff = np.max(np.abs(output_np - output_torch))
        assert np.allclose(output_np, output_torch, atol=1e-5), f"FAILED: {cfg['desc']} (max diff: {max_diff:.2e})"
        print(f"PASSED: {cfg['desc']} (max diff: {max_diff:.2e})")
    
    print("\nAll single-channel convolution tests passed!")

test_conv2d_forward()

### Visualize Edge Detection with Your Convolution

Apply the pre-defined edge detection kernels to a simple test image.

In [ ]:
# Create a simple test image with edges
test_image = np.zeros((32, 32), dtype=np.float32)
test_image[8:24, 8:24] = 1.0  # White square on black background

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Original image
axes[0, 0].imshow(test_image, cmap='gray')
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

# Apply each kernel
for idx, (name, kernel) in enumerate(KERNELS.items()):
    row = (idx + 1) // 4
    col = (idx + 1) % 4
    result = conv2d_forward(test_image, kernel, stride=1, padding=1)
    axes[row, col].imshow(result, cmap='gray')
    axes[row, col].set_title(name)
    axes[row, col].axis('off')

# Hide any unused axes
for idx in range(len(KERNELS) + 1, 8):
    axes[idx // 4, idx % 4].axis('off')

plt.suptitle('Edge Detection Kernels Applied with Your conv2d_forward', fontsize=14)
plt.tight_layout()
plt.show()

---
## Part 2: Multi-Channel Convolution (Forward Pass)

Extend your implementation to handle batched, multi-channel inputs and multiple output filters.

For a convolution with:
- Input shape: $(N, C_{in}, H, W)$
- Weight shape: $(C_{out}, C_{in}, K_h, K_w)$
- Bias shape: $(C_{out},)$

The output shape is $(N, C_{out}, H_{out}, W_{out})$ where each output channel is the sum
of convolutions across all input channels, plus the bias.

In [ ]:
def conv2d_forward_multichannel(
    input: np.ndarray,
    weight: np.ndarray,
    bias: np.ndarray = None,
    stride: int = 1,
    padding: int = 0
) -> np.ndarray:
    """Perform multi-channel 2D convolution.
    
    Args:
        input: numpy array of shape (N, C_in, H, W) -- batch of images
        weight: numpy array of shape (C_out, C_in, K_h, K_w) -- filters
        bias: numpy array of shape (C_out,) or None
        stride: int
        padding: int
        
    Returns:
        output: numpy array of shape (N, C_out, H_out, W_out)
    """
    # YOUR CODE HERE
    # Steps:
    # 1. Extract dimensions: N, C_in, H, W and C_out, C_in, K_h, K_w
    # 2. Pad the input along spatial dimensions
    # 3. Compute output spatial dimensions
    # 4. For each sample in the batch, for each output channel, sum the
    #    convolutions across all input channels
    # 5. Add bias if provided
    raise NotImplementedError("Implement conv2d_forward_multichannel")

### Test: Multi-Channel Convolution

In [ ]:
def test_conv2d_multichannel():
    """Test multi-channel convolution against PyTorch nn.Conv2d."""
    N, C_in, H, W = 2, 3, 16, 16
    C_out, K = 8, 3
    
    np.random.seed(42)
    input_np = np.random.randn(N, C_in, H, W).astype(np.float32)
    
    conv_torch = torch.nn.Conv2d(C_in, C_out, K, padding=1)
    weight_np = conv_torch.weight.detach().numpy()
    bias_np = conv_torch.bias.detach().numpy()
    
    # Your implementation
    output_np = conv2d_forward_multichannel(input_np, weight_np, bias_np, stride=1, padding=1)
    
    # PyTorch reference
    output_torch = conv_torch(torch.from_numpy(input_np)).detach().numpy()
    
    max_diff = np.max(np.abs(output_np - output_torch))
    print(f"Max absolute difference: {max_diff:.2e}")
    assert np.allclose(output_np, output_torch, atol=1e-5), "Multi-channel outputs do not match!"
    print("PASSED: Multi-channel convolution matches PyTorch")
    
    # Additional test: no bias
    conv_torch_nobias = torch.nn.Conv2d(C_in, C_out, K, padding=1, bias=False)
    weight_np2 = conv_torch_nobias.weight.detach().numpy()
    output_np2 = conv2d_forward_multichannel(input_np, weight_np2, None, stride=1, padding=1)
    output_torch2 = conv_torch_nobias(torch.from_numpy(input_np)).detach().numpy()
    max_diff2 = np.max(np.abs(output_np2 - output_torch2))
    assert np.allclose(output_np2, output_torch2, atol=1e-5), "No-bias outputs do not match!"
    print(f"PASSED: Multi-channel convolution (no bias) matches PyTorch (max diff: {max_diff2:.2e})")

test_conv2d_multichannel()

---
## Part 3: Backward Pass

Implement the backward pass for your convolution operation. This is the most educational part.

### The Math You Need

**Gradient w.r.t. bias:**

$$\frac{\partial L}{\partial b_{c_{out}}} = \sum_{n,h,w} \frac{\partial L}{\partial y_{n, c_{out}, h, w}}$$

**Gradient w.r.t. weights:**

$$\frac{\partial L}{\partial W_{c_{out}, c_{in}, m, n}} = \sum_{\text{batch}, h, w} \frac{\partial L}{\partial y_{\text{batch}, c_{out}, h, w}} \cdot x_{\text{padded}}[\text{batch}, c_{in}, h \cdot s + m, w \cdot s + n]$$

**Gradient w.r.t. input:** This is a "full" convolution of `d_output` with the flipped weight.
Each input element contributes to multiple output elements whose receptive field overlaps with
that input position. The gradient flows back from all those output positions.

In [ ]:
def conv2d_backward(
    d_output: np.ndarray,
    input: np.ndarray,
    weight: np.ndarray,
    stride: int = 1,
    padding: int = 0
) -> tuple:
    """Backward pass for 2D convolution.
    
    Args:
        d_output: numpy array of shape (N, C_out, H_out, W_out) -- gradient of loss
                  with respect to the output
        input: numpy array of shape (N, C_in, H, W) -- the original input (saved from forward)
        weight: numpy array of shape (C_out, C_in, K_h, K_w) -- the filters
        stride: int
        padding: int
        
    Returns:
        d_input: numpy array of shape (N, C_in, H, W) -- gradient w.r.t. input
        d_weight: numpy array of shape (C_out, C_in, K_h, K_w) -- gradient w.r.t. weights
        d_bias: numpy array of shape (C_out,) -- gradient w.r.t. bias
    """
    # YOUR CODE HERE
    # Steps:
    # 1. Compute d_bias: sum d_output over batch, height, width dimensions
    # 2. Pad the input (needed for d_weight computation)
    # 3. Compute d_weight: correlate d_output with padded input
    # 4. Compute d_input: "full" convolution of d_output with flipped weights
    #    (handle stride > 1 by dilating d_output)
    raise NotImplementedError("Implement conv2d_backward")

### Test: Backward Pass

Verify your gradients against PyTorch autograd. This is CRITICAL.

In [ ]:
def verify_backward(stride=1, padding=1):
    """Verify backward pass against PyTorch autograd."""
    N, C_in, H, W = 2, 3, 8, 8
    C_out, K = 4, 3
    
    np.random.seed(42)
    input_np = np.random.randn(N, C_in, H, W).astype(np.float32)
    
    # PyTorch reference
    input_torch = torch.from_numpy(input_np).requires_grad_(True)
    conv = torch.nn.Conv2d(C_in, C_out, K, stride=stride, padding=padding)
    weight_np = conv.weight.detach().numpy()
    bias_np = conv.bias.detach().numpy()
    
    output_torch = conv(input_torch)
    d_output_np = np.random.randn(*output_torch.shape).astype(np.float32)
    output_torch.backward(torch.from_numpy(d_output_np))
    
    d_input_ref = input_torch.grad.numpy()
    d_weight_ref = conv.weight.grad.numpy()
    d_bias_ref = conv.bias.grad.numpy()
    
    # Your implementation
    d_input, d_weight, d_bias = conv2d_backward(
        d_output_np, input_np, weight_np, stride=stride, padding=padding
    )
    
    # Compare
    print(f"Config: stride={stride}, padding={padding}")
    print(f"  d_input max diff:  {np.max(np.abs(d_input - d_input_ref)):.2e}")
    print(f"  d_weight max diff: {np.max(np.abs(d_weight - d_weight_ref)):.2e}")
    print(f"  d_bias max diff:   {np.max(np.abs(d_bias - d_bias_ref)):.2e}")
    
    assert np.allclose(d_input, d_input_ref, atol=1e-5), "d_input mismatch!"
    assert np.allclose(d_weight, d_weight_ref, atol=1e-5), "d_weight mismatch!"
    assert np.allclose(d_bias, d_bias_ref, atol=1e-5), "d_bias mismatch!"
    print("  PASSED!")

# Test with multiple configurations
verify_backward(stride=1, padding=1)
verify_backward(stride=1, padding=0)
verify_backward(stride=2, padding=1)
verify_backward(stride=2, padding=0)

print("\nAll backward pass tests passed!")

---
## Part 4: Benchmarking

Compare the speed of your NumPy implementation against PyTorch's `nn.Conv2d`.

In [ ]:
def benchmark(input_shape, C_out, K, stride=1, padding=0, n_runs=10):
    """Benchmark your conv vs PyTorch conv."""
    N, C_in, H, W = input_shape
    
    input_np = np.random.randn(*input_shape).astype(np.float32)
    weight_np = np.random.randn(C_out, C_in, K, K).astype(np.float32)
    bias_np = np.random.randn(C_out).astype(np.float32)
    
    # Your implementation
    start = time.time()
    for _ in range(n_runs):
        _ = conv2d_forward_multichannel(input_np, weight_np, bias_np, stride, padding)
    numpy_time = (time.time() - start) / n_runs
    
    # PyTorch CPU
    input_torch = torch.from_numpy(input_np)
    conv_torch = torch.nn.Conv2d(C_in, C_out, K, stride=stride, padding=padding)
    conv_torch.weight.data = torch.from_numpy(weight_np)
    conv_torch.bias.data = torch.from_numpy(bias_np)
    
    # Warmup
    with torch.no_grad():
        _ = conv_torch(input_torch)
    
    start = time.time()
    for _ in range(n_runs):
        with torch.no_grad():
            _ = conv_torch(input_torch)
    pytorch_time = (time.time() - start) / n_runs
    
    speedup = numpy_time / pytorch_time
    print(f"Input: {input_shape}, Conv({C_in}, {C_out}, {K}), stride={stride}, pad={padding}")
    print(f"  NumPy:   {numpy_time*1000:.2f} ms")
    print(f"  PyTorch: {pytorch_time*1000:.2f} ms")
    print(f"  Speedup: {speedup:.1f}x")
    return numpy_time, pytorch_time, speedup

In [ ]:
# Run benchmarks at multiple scales
print("=" * 60)
print("BENCHMARKS")
print("=" * 60)

results = []

# Small (LeNet-like)
print("\n--- Small (LeNet-like) ---")
results.append(benchmark((1, 1, 28, 28), 6, 5))

# CIFAR-like
print("\n--- Small (CIFAR-like) ---")
results.append(benchmark((1, 3, 32, 32), 32, 3, padding=1))

# Medium (ResNet-like)
print("\n--- Medium (ResNet-like) ---")
results.append(benchmark((1, 64, 56, 56), 64, 3, padding=1, n_runs=3))

# Large (ImageNet first conv) - reduce n_runs
print("\n--- Large (ImageNet first conv) ---")
results.append(benchmark((1, 3, 224, 224), 64, 7, stride=2, padding=3, n_runs=1))

### Benchmark Analysis Questions

Answer the following in the cells below:

1. **What is the speedup factor?** How does it vary with input size?
2. **Why is PyTorch so much faster?** (Hint: im2col, BLAS, SIMD, cuDNN)
3. **Could you make your implementation faster without calling PyTorch?**

**Your analysis here:**

*1. Speedup factor:*

YOUR ANSWER HERE

*2. Why is PyTorch faster:*

YOUR ANSWER HERE

*3. Making your implementation faster:*

YOUR ANSWER HERE

---
## Part 5: Filter Visualization

Load a pretrained CNN and visualize what its filters have learned.

In [ ]:
# Load pretrained ResNet-18
model = models.resnet18(weights='IMAGENET1K_V1')
model.eval()

# Extract first conv layer weights: (64, 3, 7, 7)
first_conv_weights = model.conv1.weight.detach().numpy()
print(f"First conv layer weight shape: {first_conv_weights.shape}")
print(f"  64 filters, each processing 3 input channels (RGB), 7x7 kernel")

# Visualize the 64 filters as RGB images
fig, axes = plt.subplots(8, 8, figsize=(16, 16))
for i, ax in enumerate(axes.flat):
    if i < 64:
        w = first_conv_weights[i]
        # Normalize each filter to [0, 1] for display
        w = (w - w.min()) / (w.max() - w.min() + 1e-8)
        # Transpose from (3, 7, 7) to (7, 7, 3) for display
        ax.imshow(w.transpose(1, 2, 0))
    ax.axis('off')
plt.suptitle("ResNet-18 First Layer Filters (64 filters, 7x7x3)", fontsize=14)
plt.tight_layout()
plt.show()

### Visualize Deeper Layer Filters and Activations

In [ ]:
# Visualize activations: feed a random image through the network
# and plot intermediate feature maps
dummy_input = torch.randn(1, 3, 224, 224)

# Hook to capture intermediate activations
activations = {}

def get_activation(name):
    def hook(model, input, output):
        activations[name] = output.detach()
    return hook

# Register hooks on key layers
model.conv1.register_forward_hook(get_activation('conv1'))
model.layer1.register_forward_hook(get_activation('layer1'))
model.layer2.register_forward_hook(get_activation('layer2'))
model.layer3.register_forward_hook(get_activation('layer3'))
model.layer4.register_forward_hook(get_activation('layer4'))

# Forward pass
with torch.no_grad():
    _ = model(dummy_input)

# Plot first 16 feature maps from each layer
fig, axes = plt.subplots(5, 16, figsize=(24, 8))
for row, (name, act) in enumerate(activations.items()):
    for col in range(16):
        if col < act.shape[1]:
            axes[row, col].imshow(act[0, col].numpy(), cmap='viridis')
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(name, fontsize=10, rotation=0, labelpad=50)

plt.suptitle('Feature Maps at Different Layers (first 16 channels)', fontsize=14)
plt.tight_layout()
plt.show()

### Filter Visualization Analysis

Answer the following:

1. **What patterns do you see in the first-layer filters?**
2. **Can you interpret deeper-layer filters? Why or why not?**
3. **What do the activation maps show at different layers?**

**Your analysis here:**

YOUR ANSWER HERE

---
## Stretch Goals (Optional)

1. **im2col optimization:** Implement the im2col transformation that converts convolution into matrix multiplication.
2. **Grouped convolution:** Support `groups > 1`. Verify against `nn.Conv2d(groups=...)`.
3. **Dilated convolution:** Add a `dilation` parameter.
4. **Transposed convolution:** Implement `ConvTranspose2d`.
5. **Numerical gradient checking:** Verify your analytical backward pass using finite differences.

In [ ]:
# Stretch goal workspace

def numerical_gradient(f, x, epsilon=1e-5):
    """Compute gradient numerically via finite differences."""
    grad = np.zeros_like(x)
    for idx in np.ndindex(x.shape):
        x_plus = x.copy()
        x_plus[idx] += epsilon
        x_minus = x.copy()
        x_minus[idx] -= epsilon
        grad[idx] = (f(x_plus) - f(x_minus)) / (2 * epsilon)
    return grad

# YOUR STRETCH GOAL CODE HERE